# Demo: Reconstruct Sensitive Features from a Facial Recognition Model

In this demonstration, we evaluate whether detailed confidence outputs from a facial recognition classifier can leak representative facial features through a model inversion workflow. The demo uses the AT&T Laboratories Cambridge face dataset, also distributed as the Olivetti faces dataset in scikit-learn.

## Scenario

A facility access system processes roughly 60,000 authentication requests per day. During an internal privacy assessment, researchers test whether repeated inference queries and probability vectors can reveal sensitive characteristics of the training distribution.

In [ ]:
from pathlib import Path
import os
import sys

import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))
os.environ.setdefault("ART_DATA_PATH", str(ROOT / "art_data"))

from model_inversion_demo_utils import (
    AttackConfig,
    art_status,
    class_prototypes,
    evaluate_predictions,
    invert_model_outputs,
    invert_model_outputs_with_face_prior,
    load_att_face_dataset,
    nearest_training_examples,
    plot_face_recovery_examples,
    plot_leakage_comparison,
    plot_reconstruction_grid,
    reconstruction_metrics,
    seed_everything,
    train_or_load_model,
    write_csv,
    write_json,
)

seed_everything(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
dataset = load_att_face_dataset(
    ROOT / "data" / "generated",
    local_att_dir=ROOT / "data" / "att_faces",
    train_per_identity=7,
    val_per_identity=3,
    validation_queries=500,
    image_size=64,
)

dataset.dataset_name, dataset.source, len(dataset.train_images), len(dataset.val_images), dataset.class_names[:5]

## Load the Facial Recognition Model

The model is intentionally compact and slightly overfit for classroom runtime. Overfit models can be more vulnerable because their decision surface may encode more training-set-specific information.

In [ ]:
model = train_or_load_model(
    ROOT / "models" / "facility_face_cnn_overfit.pt",
    dataset,
    device=device,
    epochs=50,
    batch_size=64,
    overfit=True,
)

baseline_metrics, probability_rows = evaluate_predictions(
    model,
    dataset.val_images,
    dataset.val_labels,
    dataset.class_names,
    device=device,
)
write_csv(probability_rows, ROOT / "results" / "sample_confidence_outputs.csv")
baseline_metrics, probability_rows[:3]

## Configure the Inversion Attack

The attacker repeatedly searches for an input image that makes the classifier highly confident in a target identity. Rich probability vectors provide smoother feedback than rounded confidence outputs.

In [ ]:
target_classes = list(range(6))
prototypes = class_prototypes(dataset.train_images, dataset.train_labels, target_classes)

rich_reconstructions = invert_model_outputs(
    model,
    target_classes,
    config=AttackConfig(steps=180, restarts=2, confidence_mode="rich_probability"),
    device=device,
    image_size=int(dataset.train_images.shape[-1]),
)
rounded_reconstructions = invert_model_outputs(
    model,
    target_classes,
    config=AttackConfig(steps=180, restarts=2, confidence_mode="rounded_probability"),
    device=device,
    image_size=int(dataset.train_images.shape[-1]),
)
prior_reconstructions = invert_model_outputs_with_face_prior(
    model,
    target_classes,
    dataset.val_images,
    config=AttackConfig(steps=180, restarts=2, confidence_mode="rich_probability"),
    device=device,
    n_components=80,
)
nearest_examples = nearest_training_examples(prior_reconstructions, dataset.train_images, dataset.train_labels)

metric_rows = []
metric_rows.extend(reconstruction_metrics("rich_probability", rich_reconstructions, prototypes, dataset.class_names))
metric_rows.extend(reconstruction_metrics("rounded_probability", rounded_reconstructions, prototypes, dataset.class_names))
metric_rows.extend(reconstruction_metrics("face_prior_probability", prior_reconstructions, prototypes, dataset.class_names))
write_csv(metric_rows, ROOT / "results" / "model_inversion_metrics.csv")
metric_rows[:4]

In [ ]:
summary_path = write_json(
    {
        "scenario": "Controlled facility facial recognition privacy assessment",
        "daily_authentication_requests": 60000,
        "dataset": dataset.dataset_name,
        "source": dataset.source,
        "training_images": len(dataset.train_images),
        "validation_queries": len(dataset.val_images),
        "baseline_model": baseline_metrics,
        "art_status": art_status(),
        "attack_metrics": metric_rows,
    },
    ROOT / "results" / "model_inversion_summary.json",
)
grid_path = plot_reconstruction_grid(
    rich_reconstructions,
    rounded_reconstructions,
    prototypes,
    dataset.class_names,
    ROOT / "results" / "reconstructed_feature_approximations.png",
)
face_recovery_path = plot_face_recovery_examples(
    prior_reconstructions,
    prototypes,
    nearest_examples,
    dataset.class_names,
    ROOT / "results" / "recovered_faces_from_model_inversion.png",
)
chart_path = plot_leakage_comparison(metric_rows, ROOT / "results" / "confidence_leakage_comparison.png")
summary_path, grid_path, face_recovery_path, chart_path

## Mitigation Discussion

- Return labels rather than full probability vectors when the use case permits it.
- Round, bucket, or suppress confidence scores.
- Rate-limit repeated queries and monitor identity enumeration patterns.
- Reduce overfitting and evaluate privacy leakage during model validation.
- Consider differential privacy or other privacy-preserving training methods for sensitive deployments.